In [ ]:
import os
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset, get_dataset_config_names
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm
import json

# Параметры
MODEL_NAME = "zhihan1996/DNABERT-S"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

ds = load_dataset("InstaDeepAI/nucleotide_transformer_downstream_tasks", trust_remote_code=True)

train_ds, test_ds = ds['train'], ds['test']

BATCH_SIZE = 16
PATH_TO_SAVE_OUTPUTS = "."
PARAMS_LOGREG = {'max_iter': 1000, 'random_state': 42}

class DNABERTSEmbeddingExtractor:
    """
    Класс-обёртка для извлечения эмбеддингов из модели DNABERT-S
    """
    def __init__(self, model_name=MODEL_NAME, device=DEVICE, max_length=None):
        self.device = device
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.model = AutoModel.from_pretrained(model_name).to(device)
        self.model.eval()
        # Используем max_position_embeddings из конфигурации модели, если явно не указан
        self.max_length = max_length or getattr(self.model.config, 'max_position_embeddings', 512)

    def extract_embeddings(self, sequences, batch_size=BATCH_SIZE):
        """
        Принимает список последовательностей (строк) и возвращает массив эмбеддингов [CLS]
        Размер выхода: (len(sequences), hidden_size)
        """
        all_embs = []
        for i in tqdm(range(0, len(sequences), batch_size), desc="Extracting embeddings"):
            batch = sequences[i: i + batch_size]
            enc = self.tokenizer(
                batch,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=self.max_length
            )
            enc = {k: v.to(self.device) for k, v in enc.items()}
            with torch.no_grad():
                out = self.model(**enc).last_hidden_state
                cls_emb = out[:, 0, :]
            all_embs.append(cls_emb.cpu().numpy())
            # Очистка кэша GPU для избежания OOM
            if self.device.startswith('cuda'):
                torch.cuda.empty_cache()
        return np.vstack(all_embs)


extractor = DNABERTSEmbeddingExtractor()


# Baseline: обучение на полном наборе
baseline = {}
for task in tqdm(set(train_ds['task']), desc='Baseline'):
    tr = train_ds.filter(lambda x, t=task: x['task'] == t)
    te = test_ds.filter(lambda x, t=task: x['task'] == t)
    seqs_tr, y_tr = tr['sequence'], np.array(tr['label'])
    seqs_te, y_te = te['sequence'], np.array(te['label'])


    X_tr = extractor.extract_embeddings(seqs_tr, batch_size=BATCH_SIZE)
    X_te = extractor.extract_embeddings(seqs_te, batch_size=BATCH_SIZE)

    # Обучение логистической регрессии
    clf = LogisticRegression(**PARAMS_LOGREG)
    Xf = X_tr.reshape(-1, 1) if X_tr.ndim == 1 or X_tr.shape[1] == 1 else X_tr
    Xt = X_te.reshape(-1, 1) if X_te.ndim == 1 or X_te.shape[1] == 1 else X_te
    clf.fit(Xf, y_tr)
    preds = clf.predict(Xt)

    baseline[task] = {
        'accuracy': float(accuracy_score(y_te, preds)),
        'f1_score': float(f1_score(y_te, preds, average='macro'))
    }
    with open(f'{PATH_TO_SAVE_OUTPUTS}/results_dnabert_s_task-{task}_baseline.json', 'w') as f:
        json.dump({task: baseline[task]}, f, indent=4)

# Few-shot эксперименты

def few_shot(train, test, ks=(1, 5, 10, 20), trials=5):
    res = {}
    rng = np.random.RandomState(42)
    for task in tqdm(set(train['task']), desc='Few-shot'):
        tr = train.filter(lambda x, t=task: x['task'] == t)
        te = test.filter(lambda x, t=task: x['task'] == t)
        seqs_tr, y_tr = tr['sequence'], np.array(tr['label'])
        seqs_te, y_te = te['sequence'], np.array(te['label'])
        X_te = extractor.extract_embeddings(seqs_te, batch_size=BATCH_SIZE)
        res[task] = {}
        for k in ks:
            accs, f1s = [], []
            for _ in range(trials):
                idxs = []
                for lbl in np.unique(y_tr):
                    locs = np.where(y_tr == lbl)[0]
                    choice = rng.choice(locs, size=min(k, len(locs)), replace=False)
                    idxs.extend(choice.tolist())
                X_k = extractor.extract_embeddings([seqs_tr[i] for i in idxs], batch_size=BATCH_SIZE)
                y_k = y_tr[idxs]
                clf = LogisticRegression(**PARAMS_LOGREG)
                Xf = X_k.reshape(-1, 1) if X_k.ndim == 1 or X_k.shape[1] == 1 else X_k
                Xt = X_te.reshape(-1, 1) if X_te.ndim == 1 or X_te.shape[1] == 1 else X_te
                clf.fit(Xf, y_k)
                p = clf.predict(Xt)
                accs.append(accuracy_score(y_te, p))
                f1s.append(f1_score(y_te, p, average='macro'))
            res[task][k] = {'accuracy': float(np.mean(accs)), 'f1_score': float(np.mean(f1s))}
            with open(f'{PATH_TO_SAVE_OUTPUTS}/results_dnabert_s_task-{task}_k-{k}.json', 'w') as f:
                json.dump({task: res[task][k]}, f, indent=4)
    return res

results_kshot = few_shot(train_ds, test_ds)

output = {'full': baseline, 'kshot': results_kshot, 'params': PARAMS_LOGREG}
with open(f'{PATH_TO_SAVE_OUTPUTS}/results_dnabert_s.json', 'w') as f:
    json.dump(output, f, indent=4)


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'InstaDeepAI/nucleotide_transformer_downstream_tasks' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
Some weights of the model checkpoint at zhihan1996/DNABERT-S were not used when initializing BertModel: ['encoder.layer.0.attention.self.Wqkv.bias', 'encoder.layer.0.attention.self.Wqkv.weight', 'encoder.layer.0.mlp.gated_layers.weight', 'encoder.layer.0.mlp.layernorm.bias', 'encoder.layer.0.mlp.layernorm.weight', 'encoder.layer.0.mlp.wo.bias', 'encoder.layer.0.mlp.wo.weight', 'encoder.layer.1.attention.self.Wqkv.bias', 'encoder.layer.1.attention.self.Wqkv.weight', 'encoder.layer.1.mlp.gated_layers.weight', 'encoder.layer.1.mlp.layernorm.bias', 'encoder.layer.1.mlp.layernorm.weight', 'encoder.layer.1.mlp.wo.bias', 'encoder.layer.1.mlp.wo.weight', 